### Same whatever we have done for GPT in notebook 6, 7 and 8 we will do for BERT

In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F

VOCAB_SIZE = 5000

EMBED_DIM = 256

MAX_LEN = 128

NUM_HEADS = 8

NUM_LAYERS = 6

DROPOUT = 0.1

In [3]:
import importlib

# 1. Use importlib to dynamically load the notebook 5 module
notebook05_module = importlib.import_module("ipynb.fs.full.05_TransformerBlock(BERT Encoder Block)")

# 2. Extract the class from the loaded module
BertEncoderBlock = notebook05_module.BertEncoderBlock


torch.Size([2, 10, 768])


In [4]:
x = torch.randn(
    2,
    128,
    EMBED_DIM
)

block = BertEncoderBlock(
    EMBED_DIM,
    NUM_HEADS
)

out = block(x)

print(out.shape)

torch.Size([2, 128, 256])


In [5]:
class MiniBERT(nn.Module):

    def __init__(
        self,
        vocab_size,
        embed_dim,
        max_len,
        num_heads,
        num_layers
    ):

        super().__init__()

        self.token_embedding = nn.Embedding(
            vocab_size,
            embed_dim
        )

        self.position_embedding = nn.Embedding(
            max_len,
            embed_dim
        )

        self.segment_embedding = nn.Embedding(
            2,
            embed_dim
        )

        self.encoder_blocks = nn.Sequential(

            *[
                BertEncoderBlock(
                    embed_dim,
                    num_heads
                )

                for _ in range(
                    num_layers
                )
            ]

        )

        self.ln_f = nn.LayerNorm(
            embed_dim
        )

        self.mlm_head = nn.Linear(
            embed_dim,
            vocab_size
        )

    def forward(

        self,

        input_ids,

        segment_ids=None
    ):

        B,T = input_ids.shape

        if segment_ids is None:

            segment_ids = torch.zeros_like(
                input_ids
            )

        token_embeddings = (
            self.token_embedding(
                input_ids
            )
        )

        position_embeddings = (
            self.position_embedding(
                torch.arange(
                    T,
                    device=input_ids.device
                )
            )
        )

        segment_embeddings = (
            self.segment_embedding(
                segment_ids
            )
        )

        x = (

            token_embeddings

            +

            position_embeddings

            +

            segment_embeddings

        )

        x = self.encoder_blocks(
            x
        )

        x = self.ln_f(
            x
        )

        logits = self.mlm_head(
            x
        )

        return logits

In [6]:
model = MiniBERT(

    vocab_size=VOCAB_SIZE,

    embed_dim=EMBED_DIM,

    max_len=MAX_LEN,

    num_heads=NUM_HEADS,

    num_layers=NUM_LAYERS

)

In [7]:
total_params = sum(

    p.numel()

    for p in model.parameters()

)

print(
    f"{total_params:,}"
)

7,332,744


In [8]:
x = torch.randint(

    0,

    VOCAB_SIZE,

    (4,128)

)

logits = model(
    x
)

print(
    logits.shape
)

torch.Size([4, 128, 5000])
